# Task 2 ensemble: which of two books did the user read?

Validation pairs are built like the test pairs: a held-out interaction against the book of a random other interaction
(drawn in proportion to popularity). Sub-models score (user, book); a LightGBM pair blender decides every pair.
The Task 2 test file has no timestamp.

- **popularity**: log(1 + #interactions of the book)
- **item cosine**: similarity of the book to the books the user has read
- **EASE**: linear autoencoder in closed form (Steck 2019), all books, computed in RAM
- **Mult-VAE**: non-linear variational autoencoder (Liang et al. 2018), trained on the GPU with early stopping

Every sub-model cell stores its scores in `trained_models/task2/<model>.pt`; the blend cells only need the stored files.
All hyperparameters: `submission/lib/config.py`, Task 2 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.config import TASK2_PAIR_BLENDER
from lib.models.ease_model import load_ease_model, load_time_ease_model, store_ease_model, store_time_ease_model
from lib.models.item_cosine_model import load_item_cosine_model, store_item_cosine_model
from lib.models.mult_vae import load_mult_vae, store_mult_vae
from lib.models.pair_blender import load_pair_blender, store_pair_blender
from lib.models.popularity_model import load_popularity_model, store_popularity_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


In [2]:
# training file -> validation pairs (held-out interactions vs. popularity-drawn negatives) + the test pairs
task2 = assignment1.load_task2()

## Sub-models

In [3]:
# popularity: log(1 + #interactions of the book) (seconds)
popularity_result = assignment1.train_task2_popularity_model(task2)
store_popularity_model(popularity_result, "task2")

[01:30:06] popularity | validation accuracy 0.4914
[01:30:06] stored popularity -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task2\popularity.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task2/popularity.pt')

In [4]:
# item cosine similarity to the user's history (CPU, about a minute)
item_cosine_result = assignment1.train_task2_item_cosine_model(task2)
store_item_cosine_model(item_cosine_result, "task2")

[01:30:26] item cosine | validation accuracy 0.8328
[01:30:49] stored item_cosine -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task2\item_cosine.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task2/item_cosine.pt')

In [5]:
# EASE: one Gram matrix + one matrix inverse per fit (all ~51k books in RAM, a few minutes per fit); only the scores are stored, not the dense weights
ease_result = assignment1.train_task2_ease_model(task2)
store_ease_model(ease_result, "task2")

[01:33:24] EASE | validation accuracy 0.8491
[01:35:59] stored ease -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task2\ease.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task2/ease.pt')

In [6]:
# Mult-VAE: non-linear autoencoder, early stopping on the validation pair accuracy, then refit on all interactions (GPU)
mult_vae_result = assignment1.train_task2_mult_vae(task2)
store_mult_vae(mult_vae_result, "task2")

[01:36:05] Mult-VAE epoch   1 | train loss 558.92 | validation accuracy 0.5765
[01:36:10] Mult-VAE epoch   2 | train loss 526.19 | validation accuracy 0.7427
[01:36:15] Mult-VAE epoch   3 | train loss 505.26 | validation accuracy 0.7878
[01:36:20] Mult-VAE epoch   4 | train loss 492.10 | validation accuracy 0.8153
[01:36:25] Mult-VAE epoch   5 | train loss 484.52 | validation accuracy 0.8248
[01:36:29] Mult-VAE epoch   6 | train loss 479.48 | validation accuracy 0.8322
[01:36:34] Mult-VAE epoch   7 | train loss 475.16 | validation accuracy 0.8380
[01:36:39] Mult-VAE epoch   8 | train loss 470.50 | validation accuracy 0.8425
[01:36:44] Mult-VAE epoch   9 | train loss 466.87 | validation accuracy 0.8472
[01:36:49] Mult-VAE epoch  10 | train loss 463.47 | validation accuracy 0.8488
[01:36:53] Mult-VAE epoch  11 | train loss 459.80 | validation accuracy 0.8492
[01:36:58] Mult-VAE epoch  12 | train loss 456.14 | validation accuracy 0.8514
[01:37:03] Mult-VAE epoch  13 | train loss 452.97 | 

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task2/mult_vae.pt')

## Blend

In [7]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "popularity": load_popularity_model,
    "item_cosine": load_item_cosine_model,
    "ease": load_ease_model,
    "time_ease": load_time_ease_model,
    "mult_vae": load_mult_vae,
}
task2_model_results = {model_name: sub_model_loaders[model_name]("task2") for model_name in TASK2_PAIR_BLENDER.combined_models}

In [8]:
# LightGBM pair blender: P(item1 is the real one) from both books' scores + meta-features (prints the validation accuracies)
pair_blend_result = assignment1.train_task2_blender(task2, task2_model_results)
store_pair_blender(pair_blend_result, "task2")

[01:39:40] popularity             validation accuracy 0.4914
[01:39:40] item_cosine            validation accuracy 0.8328
[01:39:40] ease                   validation accuracy 0.8491
[01:39:40] mult_vae               validation accuracy 0.8562
[01:39:40] pair blend             validation accuracy 0.9397  (out-of-fold, 5 folds)
[01:39:41] pair blend inputs: popularity_difference 1600, mult_vae_difference 1410, ease_difference 1206, popularity_first 824, popularity_second 820, log_user_interactions 799, item_cosine_difference 764, item_cosine_second 736, item_cosine_first 717, ease_second 685, ease_first 672, mult_vae_second 670, mult_vae_first 633, already_read_difference 464, already_read_first 0, already_read_second 0
[01:39:41] stored pair_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task2\pair_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task2/pair_blender.pt')

In [9]:
# writes submission/output/task_2_predictions.csv (the "which" column = the chosen book)
submitted_blend_result = load_pair_blender("task2")
assignment1.write_task2_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK2_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_2_predictions.csv
